In [1]:
# 5a_derive_variables.ipynb
#
# Reads feature-engineered UKHLS (step 4a), adds composite columns:
#   o_digital_use — mean of mapped 0–1 scores over netpusenew, laptop, smtphone,
#                   browse, email, smlook, smpost, onlinebuy, onlinebank, streaming
#   o_service_use — mean over hl2gp, servuse2/10, benefit receipts, fimnsben_dv
#   o_derived_work_status — copies jbnssec8_dv (column o_jbnssec8_dv) when raw jbstat is 1 or 2; else NaN
#
# Raw jbstat comes from step 3a backfill (merged on pidp) so we can distinguish 1 vs 2 before recode.
# Logic lives in helpers/derive_variables.py (single source for scoring rules).
# Output feeds 6a_synthetic_population.ipynb.

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
from data_pipeline.config_paths import DATA_FOLDER
import data_pipeline.config_variables as _cv
_cv.reload_config_variables()

import pandas as pd

from data_pipeline.helpers import derive_variables
importlib.reload(derive_variables)
from data_pipeline.helpers.derive_variables import (
    compute_digital_use,
    compute_service_use,
    compute_derived_work_status,
)

WAVE = "o"
INPUT_PKL   = f"../{DATA_FOLDER}/4_feature_eng_ukhls/o_indresp_feature_eng.pkl"
BACKFILL_PKL = f"../{DATA_FOLDER}/3_backfill_ukhls_waves/o_indresp_backfilled.pkl"
OUTPUT_DIR  = f"../{DATA_FOLDER}/5_derive_variables"
OUTPUT_PKL  = f"{OUTPUT_DIR}/o_indresp_derived.pkl"

os.makedirs(OUTPUT_DIR, exist_ok=True)
if not os.path.exists(INPUT_PKL):
    raise FileNotFoundError(f"{INPUT_PKL} not found — run 4a_feature_eng_ukhls.ipynb first.")

df = pd.read_pickle(INPUT_PKL)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns from {INPUT_PKL}")

if os.path.exists(BACKFILL_PKL):
    jb = pd.read_pickle(BACKFILL_PKL)[["pidp", f"{WAVE}_jbstat"]].rename(
        columns={f"{WAVE}_jbstat": f"{WAVE}_jbstat_raw"}
    )
    df = df.merge(jb, on="pidp", how="left")
else:
    print(f"  WARNING: {BACKFILL_PKL} not found — derived_work_status uses recoded jbstat only.")

df[f"{WAVE}_digital_use"] = compute_digital_use(df, wave=WAVE)
df[f"{WAVE}_service_use"] = compute_service_use(df, wave=WAVE)
df[f"{WAVE}_derived_work_status"] = compute_derived_work_status(df, wave=WAVE)

drop_raw = f"{WAVE}_jbstat_raw"
if drop_raw in df.columns:
    df.drop(columns=[drop_raw], inplace=True)

print(
    f"  {WAVE}_digital_use:  min={df[f'{WAVE}_digital_use'].min():.3f}  "
    f"max={df[f'{WAVE}_digital_use'].max():.3f}  mean={df[f'{WAVE}_digital_use'].mean():.3f}"
)
print(
    f"  {WAVE}_service_use: min={df[f'{WAVE}_service_use'].min():.3f}  "
    f"max={df[f'{WAVE}_service_use'].max():.3f}  mean={df[f'{WAVE}_service_use'].mean():.3f}"
)
_dw = df[f"{WAVE}_derived_work_status"]
print(
    f"  {WAVE}_derived_work_status: non-null={_dw.notna().sum():,}  "
    f"min={_dw.min():.3f}  max={_dw.max():.3f}  mean={_dw.mean():.3f}"
)

df.to_pickle(OUTPUT_PKL, protocol=5)
print(f"\nSaved {OUTPUT_PKL}")
print(f"Shape: {df.shape}")



🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  

Loaded 19,618 rows × 62 columns from ../data/4_feature_eng_ukhls/o_indresp_feature_eng.pkl
  o_digital_use:  min=0.000  max=1.000  mean=0.674
  o_service_use: min=0.000  max=1.000  mean=0.260
  o_derived_work_status: non-null=10,827  min=0.000  max=8.000  mean=3.676

Saved ../data/5_derive_variables/o_indresp_derived.pkl
Shape: (19618, 65)
